[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nursnaaz/zero-to-genai-engineer/blob/main/10_RAG/notebooks/10_deepeval_evaluation.ipynb)

# Evaluating RAG with DeepEval

**Notebook 10 · Phase 3 (Evaluation)** · Stack: `deepeval`, `langchain-openai`

Notebook 09 used **RAGAS**. **DeepEval** is the other major LLM-eval framework — same core RAG
metrics, but built for **software testing**: every metric has a **threshold** (pass/fail), a
**human-readable reason**, and first-class **pytest** integration so evals run in CI like unit
tests. It also adds **GEval** (define your own metric in plain English) and red-teaming.

We reuse the same four planted-defect examples as Notebook 09 so you can compare the two
frameworks directly.

| DeepEval RAG metric | Component | Analogue in RAGAS |
|---------------------|-----------|-------------------|
| `FaithfulnessMetric` | generator | Faithfulness |
| `AnswerRelevancyMetric` | generator | ResponseRelevancy |
| `ContextualPrecisionMetric` | retriever | LLMContextPrecision |
| `ContextualRecallMetric` | retriever | ContextRecall |
| `ContextualRelevancyMetric` | retriever | (aspect of context precision) |

## 0. Install dependencies

Run first (idempotent). Restart the kernel once after a fresh install.

In [1]:
%pip install -q "numpy<2" deepeval langchain-openai python-dotenv
print("✅ Dependencies ready. If this was a fresh install, restart the kernel, then re-run.")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /Users/mohamednoordeenalaudeen/Documents/GenAI-2026/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ Dependencies ready. If this was a fresh install, restart the kernel, then re-run.


## 1. Setup

Three practical settings for a clean, reliable run:
- **`DEEPEVAL_TELEMETRY_OPT_OUT`** — no telemetry / login prompts.
- **`async_mode=False`** on metrics — deterministic synchronous execution in a notebook.
- **Judge = `gpt-4o`** — DeepEval uses *structured outputs*; small models (gpt-4o-mini) can
  occasionally run away to the token limit on the schema, so we use a sturdier judge here.

In [2]:
import warnings, os
warnings.filterwarnings("ignore")
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"
import logging
for _n in ("httpx","openai","httpcore"): logging.getLogger(_n).setLevel(logging.ERROR)
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")

import deepeval
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (FaithfulnessMetric, AnswerRelevancyMetric,
                              ContextualPrecisionMetric, ContextualRecallMetric)
JUDGE = "gpt-4o"
def faithfulness(): return FaithfulnessMetric(threshold=0.7, model=JUDGE, include_reason=True, async_mode=False)
def answer_rel():   return AnswerRelevancyMetric(threshold=0.7, model=JUDGE, include_reason=True, async_mode=False)
def ctx_prec():     return ContextualPrecisionMetric(threshold=0.7, model=JUDGE, include_reason=True, async_mode=False)
def ctx_recall():   return ContextualRecallMetric(threshold=0.7, model=JUDGE, include_reason=True, async_mode=False)
print("DeepEval", deepeval.__version__, "ready — judge =", JUDGE)

DeepEval 4.0.7 ready — judge = gpt-4o


## 2. The `LLMTestCase`

DeepEval's unit of evaluation. Fields mirror a RAG interaction:
- `input` — the question
- `actual_output` — what your app generated
- `retrieval_context` — the chunks retrieved
- `expected_output` — ground truth (needed for contextual precision/recall)

## 3. Same four examples, one planted defect each

In [3]:
cases = {
 "A. all good": LLMTestCase(
    input="What port does HTTPS use by default?",
    actual_output="HTTPS uses TCP port 443 by default.",
    expected_output="443",
    retrieval_context=["HTTPS uses TCP port 443 by default.", "HTTP uses TCP port 80 by default."]),
 "B. hallucination": LLMTestCase(
    input="How do I undo the last commit but keep changes staged?",
    actual_output="Run git reset --hard HEAD~1, which keeps your changes staged.",
    expected_output="git reset --soft HEAD~1",
    retrieval_context=["git reset --soft HEAD~1 undoes the last commit while keeping its changes staged."]),
 "C. off-topic": LLMTestCase(
    input="What causes a myocardial infarction?",
    actual_output="Heart disease is serious, so exercise regularly and eat a balanced diet.",
    expected_output="Occlusion of a coronary artery cutting off blood flow to the heart muscle.",
    retrieval_context=["A myocardial infarction is caused by occlusion of a coronary artery, cutting blood flow to heart muscle."]),
 "D. bad retrieval": LLMTestCase(
    input="What port does SSH use by default?",
    actual_output="SSH uses TCP port 22 by default.",
    expected_output="22",
    retrieval_context=["HTTPS uses port 443.", "HTTP uses port 80.", "DNS uses port 53."]),
}
print(len(cases), "test cases")

4 test cases


In [4]:
# Measure all four metrics on every case; store score + pass/fail + reason.
results = {}
for name, tc in cases.items():
    row = {}
    for key, make in [("faithful", faithfulness), ("ans_rel", answer_rel),
                      ("ctx_prec", ctx_prec), ("ctx_recall", ctx_recall)]:
        m = make(); m.measure(tc)
        row[key] = (round(m.score, 2), m.is_successful(), m.reason)
    results[name] = row

hdr = f"{'case':18} {'faithful':>16} {'ans_rel':>16} {'ctx_prec':>16} {'ctx_recall':>16}"
print(hdr); print("-"*len(hdr))
for name, row in results.items():
    cells_ = [f"{v[0]:.2f} {'PASS' if v[1] else 'FAIL'}" for v in row.values()]
    print(f"{name:18} " + " ".join(f"{c:>16}" for c in cells_))

Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

case                       faithful          ans_rel         ctx_prec       ctx_recall
--------------------------------------------------------------------------------------
A. all good               1.00 PASS        1.00 PASS        1.00 PASS        1.00 PASS
B. hallucination          0.00 FAIL        0.50 FAIL        1.00 PASS        1.00 PASS
C. off-topic              1.00 PASS        0.33 FAIL        1.00 PASS        1.00 PASS
D. bad retrieval          1.00 PASS        1.00 PASS        0.00 FAIL        0.00 FAIL


Each planted defect drops the matching metric **below its 0.7 threshold → FAIL**, exactly like
RAGAS. DeepEval's addition is the explicit **pass/fail** verdict per metric.

## 4. DeepEval's edge #1 — human-readable reasons

`include_reason=True` makes every metric explain *why* it scored what it did. Invaluable for
debugging a failing eval (and for non-ML stakeholders).

In [5]:
for name in ["B. hallucination", "D. bad retrieval"]:
    print(f"[{name}]")
    print("  faithfulness :", results[name]["faithful"][0], "->", results[name]["faithful"][2][:150])
    print("  ctx_recall   :", results[name]["ctx_recall"][0], "->", results[name]["ctx_recall"][2][:150])
    print()

[B. hallucination]
  faithfulness : 0.0 -> The score is 0.00 because the actual output incorrectly states that 'git reset --hard HEAD~1' keeps changes staged, while it actually discards all cha
  ctx_recall   : 1.0 -> The score is 1.00 because the sentence 'git reset --soft HEAD~1' is perfectly aligned with the information in the 1st node in the retrieval context, e

[D. bad retrieval]
  faithfulness : 1.0 -> The score is 1.00 because there are no contradictions between the actual output and the retrieval context. Everything aligns perfectly, showcasing a h
  ctx_recall   : 0.0 -> The score is 0.00 because the expected output '22' does not align with any node(s) in the retrieval context, which only references ports 443, 80, and 



## 5. DeepEval's edge #2 — custom criteria with `GEval`

Not every requirement fits a prebuilt metric. **GEval** lets you define one in plain English
(it builds an LLM-judge rubric). Here: *does the answer stay strictly within the retrieved
context?* — a citation/grounding check tuned to our own standard.

In [6]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

grounding = GEval(
    name="Strict Grounding",
    criteria=("Return 1 only if EVERY factual claim in 'actual output' is directly supported by "
              "'retrieval context'. Penalize any claim that is not present in the context."),
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT,
                       LLMTestCaseParams.RETRIEVAL_CONTEXT],
    model=JUDGE, threshold=0.7, async_mode=False,
)
for name in ["A. all good", "B. hallucination"]:
    grounding.measure(cases[name])
    print(f"{name:18} GEval(Strict Grounding) = {grounding.score:.2f}  {'PASS' if grounding.is_successful() else 'FAIL'}")
    print("   reason:", grounding.reason[:160], "\n")

Output()

Output()

A. all good        GEval(Strict Grounding) = 1.00  PASS
   reason: The actual output correctly states that HTTPS uses TCP port 443 by default, which is directly supported by the retrieval context. There is full alignment betwee 



B. hallucination   GEval(Strict Grounding) = 0.00  FAIL
   reason: The actual output incorrectly suggests using 'git reset --hard HEAD~1', which does not keep changes staged, contradicting the retrieval context that specifies ' 



## 6. DeepEval's edge #3 — evals as **pytest** tests (CI-native)

DeepEval is designed to run in CI. You wrap a metric check in a test with `assert_test`, then
`deepeval test run test_rag.py` fails the build on regressions — evals become unit tests.

```python
# test_rag.py
import pytest
from deepeval import assert_test
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric

@pytest.mark.parametrize("tc", load_test_cases())   # your golden set
def test_faithfulness(tc):
    assert_test(tc, [FaithfulnessMetric(threshold=0.7, model="gpt-4o")])
```

The same `is_successful()` verdict we used above becomes a red/green build signal. *(Shown as a
reference — running pytest is outside this notebook.)*

## 7. RAGAS vs. DeepEval — when to use which

| | **RAGAS** (Notebook 09) | **DeepEval** (this notebook) |
|---|---|---|
| Mental model | data-centric evaluation | software testing (pytest) |
| Output | scores | scores + **pass/fail thresholds** + **reasons** |
| CI integration | via scripts | **first-class** (`deepeval test run`) |
| Custom metrics | custom metrics API | **GEval** (plain-English criteria) |
| Test-set generation | **strong** (synthetic Q/A from docs) | supported |
| Extras | research metrics, optimization | **red-teaming**, guardrails |
| Metric definitions | *faithfulness = claims supported by context* | *faithfulness = claims not contradicted by context* |

> **Note the last row** — we saw it live: on "D. bad retrieval", RAGAS scored faithfulness low
> (the SSH answer isn't *supported* by the HTTPS/HTTP/DNS context), while DeepEval scored it high
> (nothing *contradicts* it). Same word, different definition. Always read a framework's metric
> definition before trusting the number.

**Rule of thumb:** DeepEval when evals live in your **CI/test suite** and you want pass/fail gates
+ reasons; RAGAS when you're doing **data-centric** measurement, test-set generation, or research
metrics. Many teams use both.

## 8. Summary

- DeepEval measures the same RAG failure modes as RAGAS, but as **thresholded pass/fail tests**
  with **explanatory reasons**.
- **GEval** turns a plain-English rule into a custom metric; **pytest integration** turns evals
  into CI gates.
- Metric *names* match across frameworks but *definitions* can differ (faithfulness) — read the docs.
- Use `async_mode=False` and a sturdy judge (`gpt-4o`) for reliable notebook runs.

### Next — the capstone
With both evaluators in hand, the final notebook assembles the **entire pipeline** (ingest →
chunk → embed → index → hybrid retrieve → rerank → generate cited answer) and uses these metrics
to prove each stage improves the end-to-end system.